# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub pyarrow pandas

In [2]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

In [3]:
con.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    INSTALL parquet;
    LOAD parquet;
""")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con.sql(f"""
    CREATE OR REPLACE VIEW fact_content_daily_performance AS
    SELECT *
    FROM read_parquet('{march_path}');
""")

print("March 2026 warehouse data connected.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March 2026 warehouse data connected.


## 1. My rule and its reason codes

### Rule

Prioritize content that has meaningful search visibility, ranks within the top 10 positions, and has a low click-through rate (CTR). These pages have enough search exposure to justify review and may have an opportunity to improve clicks.

### Reason code

- `HIGH_VISIBILITY_LOW_CTR` — The content has at least 500 impressions, an average position within the top 10, and CTR below 0.5%.

### Action

- `Review CTR` — Prioritize the page for content/SEO review focused on improving click-through performance.

In [6]:
baseline = con.sql("""
WITH content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
scored AS (
    SELECT
        *,
        100.0 * clicks / NULLIF(impressions, 0) AS ctr_pct
    FROM content_month
    WHERE impressions > 0
      AND avg_position > 0
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    ROUND(avg_position, 2) AS avg_position,
    ROUND(ctr_pct, 2) AS ctr_pct,

    CASE
        WHEN impressions >= 500
         AND avg_position <= 10
         AND ctr_pct < 0.5
        THEN 1
        ELSE 0
    END AS score,

    CASE
        WHEN impressions >= 500
         AND avg_position <= 10
         AND ctr_pct < 0.5
        THEN 'HIGH_VISIBILITY_LOW_CTR'
        ELSE 'NO_PRIORITY_SIGNAL'
    END AS reason_code,

    CASE
        WHEN impressions >= 500
         AND avg_position <= 10
         AND ctr_pct < 0.5
        THEN 'Review CTR'
        ELSE 'Monitor'
    END AS action

FROM scored
ORDER BY score DESC, impressions DESC
""").df()

baseline.head(20)


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr_pct,score,reason_code,action
0,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.51,0.33,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.67,0.01,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
2,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.30,0.42,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
3,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.47,0.14,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,4.55,0.19,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
5,client_e547b89c05043229,content_4ffe18112a5642e3,186983.0,586.0,2.39,0.31,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
6,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,3.40,0.15,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
7,client_73cda7b4e4f265ea,content_471d9cabce329a66,164885.0,396.0,4.60,0.24,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
8,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,151166.0,408.0,3.43,0.27,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
9,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.17,0.03,1,HIGH_VISIBILITY_LOW_CTR,Review CTR


In [4]:
volume_audit = con.sql("""
WITH content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN impressions < 100 THEN '0-99'
        WHEN impressions < 500 THEN '100-499'
        WHEN impressions < 1000 THEN '500-999'
        WHEN impressions < 5000 THEN '1000-4999'
        ELSE '5000+'
    END AS impression_bucket,
    COUNT(*) AS n,
    ROUND(AVG(impressions), 1) AS avg_impressions,
    ROUND(
        100.0 * SUM(clicks) / NULLIF(SUM(impressions), 0),
        2
    ) AS overall_ctr_pct
FROM content_month
GROUP BY 1
ORDER BY
    CASE impression_bucket
        WHEN '0-99' THEN 1
        WHEN '100-499' THEN 2
        WHEN '500-999' THEN 3
        WHEN '1000-4999' THEN 4
        ELSE 5
    END
""").df()

volume_audit

,impression_bucket,n,avg_impressions,overall_ctr_pct
0,0-99,75297,24.7,0.35
1,100-499,39517,250.0,0.23
2,500-999,16866,716.7,0.25
3,1000-4999,31766,2391.8,0.30
4,5000+,13292,13605.9,0.29


In [5]:
position_audit = con.sql("""
WITH content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
position_data AS (
    SELECT
        *,
        100.0 * clicks / NULLIF(impressions, 0) AS ctr_pct
    FROM content_month
    WHERE impressions > 0
      AND avg_position > 0
)
SELECT
    CASE
        WHEN avg_position <= 3 THEN '1-3'
        WHEN avg_position <= 5 THEN '4-5'
        WHEN avg_position <= 10 THEN '6-10'
        WHEN avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(impressions), 1) AS avg_impressions,
    ROUND(AVG(ctr_pct), 2) AS avg_ctr_pct
FROM position_data
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-5' THEN 2
        WHEN '6-10' THEN 3
        WHEN '11-20' THEN 4
        ELSE 5
    END
""").df()

position_audit

,position_bucket,n,avg_impressions,avg_ctr_pct
0,1-3,17426,2376.8,1.00
1,4-5,27712,2722.9,0.62
2,6-10,55576,1307.3,0.42
3,11-20,29922,1042.4,0.33
4,21+,44668,1341.8,0.20


### Signal audit verdicts

- **Search volume — MIXED:** Higher-impression content represents greater visible opportunity, but CTR does not increase consistently across volume buckets. This supports using volume for prioritization, but not as a standalone quality signal.

- **CTR vs position — CONFIRMED:** CTR decreases consistently as average search position gets worse, from 1.00% for positions 1–3 to 0.20% for positions 21+. This supports using search position when prioritizing low-CTR content.

### Baseline decision

The baseline combines the two audited signals: meaningful search visibility and low CTR at a top-10 average position. The rule is intended as a transparent prioritization baseline rather than a trained predictive model.

## 2. Build the ranked queue (writes CSV)

The baseline score ranks content using the transparent rule defined above. Higher-scoring content is placed first, followed by higher-impression content.

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

In [7]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked_queue = baseline.sort_values(
    by=["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

ranked_queue["rank"] = ranked_queue.index + 1

output_path = "work/outputs/baseline_action_score.csv"

ranked_queue.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(ranked_queue):,}")

ranked_queue.head(20)

Saved: work/outputs/baseline_action_score.csv
Rows: 175,304


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr_pct,score,reason_code,action,rank
0,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.51,0.33,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,1
1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.67,0.01,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,2
2,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.30,0.42,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,3
3,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.47,0.14,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,4
4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,4.55,0.19,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,5
5,client_e547b89c05043229,content_4ffe18112a5642e3,186983.0,586.0,2.39,0.31,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,6
6,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,3.40,0.15,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,7
7,client_73cda7b4e4f265ea,content_471d9cabce329a66,164885.0,396.0,4.60,0.24,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,8
8,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,151166.0,408.0,3.43,0.27,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,9
9,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.17,0.03,1,HIGH_VISIBILITY_LOW_CTR,Review CTR,10


## 3. Top-20 review

The top 20 ranked items are reviewed manually. Each review records the recommended action, the reason for the ranking, a confidence note, and what could make the recommendation wrong.

In [8]:
top20 = ranked_queue.head(20).copy()

top20[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "avg_position",
    "ctr_pct",
    "score",
    "reason_code",
    "action"
]]


,rank,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr_pct,score,reason_code,action
0,1,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.51,0.33,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
1,2,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.67,0.01,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
2,3,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.30,0.42,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
3,4,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.47,0.14,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
4,5,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,4.55,0.19,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
5,6,client_e547b89c05043229,content_4ffe18112a5642e3,186983.0,586.0,2.39,0.31,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
6,7,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,3.40,0.15,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
7,8,client_73cda7b4e4f265ea,content_471d9cabce329a66,164885.0,396.0,4.60,0.24,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
8,9,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,151166.0,408.0,3.43,0.27,1,HIGH_VISIBILITY_LOW_CTR,Review CTR
9,10,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.17,0.03,1,HIGH_VISIBILITY_LOW_CTR,Review CTR


### Top-10 manual review

| Rank | Action | Why it is here | Confidence | What would make it wrong |
|---|---|---|---|---|
| 1 | Review CTR | 221,310 impressions, position 2.51, but CTR is only 0.33%. Very high visibility with underperforming clicks. | High | CTR may be explained by search intent or SERP features. |
| 2 | Review CTR | 212,404 impressions and position 0.67, but CTR is only 0.01%. Extremely low CTR despite very strong visibility. | High | The unusually low CTR may reflect data/query characteristics or SERP behavior. |
| 3 | Review CTR | 205,867 impressions, position 3.30, and CTR of 0.42%. Meets the rule while having substantial visibility. | Medium | Search intent or SERP features could explain the lower-than-expected clicks. |
| 4 | Review CTR | 203,497 impressions, position 2.47, and CTR of 0.14%. Strong visibility with a notably low CTR. | High | The low CTR may be caused by query mix or SERP features rather than the content itself. |
| 5 | Review CTR | 194,337 impressions, position 4.55, and CTR of 0.19%. Good visibility and top-10 position with low CTR. | Medium | Position is weaker than the highest-ranked items, so ranking limitations may contribute. |
| 6 | Review CTR | 186,983 impressions, position 2.39, and CTR of 0.31%. High visibility and strong position but relatively low CTR. | Medium | Search intent, SERP layout, or seasonal effects could affect CTR. |
| 7 | Review CTR | 170,808 impressions, position 3.40, and CTR of 0.15%. Strong search position combined with low CTR. | High | The observed CTR may reflect query mix or SERP features. |
| 8 | Review CTR | 164,885 impressions, position 4.60, and CTR of 0.24%. Sufficient visibility and top-10 position trigger the rule. | Medium | The page may need ranking improvement rather than a CTR-focused change. |
| 9 | Review CTR | 151,166 impressions, position 3.43, and CTR of 0.27%. Strong visibility and position with low CTR. | Medium | Search intent or SERP features may explain the observed CTR. |
| 10 | Review CTR | 143,019 impressions, position 3.17, but CTR is only 0.03%. Very strong position and visibility with extremely low CTR. | High | The unusually low CTR could be caused by query mix or other SERP behavior. |

## 4. Weak picks + leakage check

### Weak picks

The baseline may produce weak picks when low CTR is caused by factors that are not actionable through content changes, such as search intent, SERP features, seasonality, or unusual query mix.

### Leakage check

The rule uses only March 2026 observed search performance:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`

No future-window data, decline labels, trend labels, or label-derived features are used.

In [10]:
# Explicit leakage check for baseline inputs

baseline_inputs = {
    "gsc_impressions",
    "gsc_clicks",
    "avg_position",
    "ctr_pct"
}

forbidden_terms = [
    "decline",
    "trend",
    "label",
    "target",
    "future",
    "proxy"
]

leakage_candidates = [
    col for col in baseline.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Baseline inputs:")
print(sorted(baseline_inputs))

print("\nPotential leakage-related columns present in baseline:")
print(leakage_candidates)

print("\nLeakage check:")
if any(
    any(term in col.lower() for term in forbidden_terms)
    for col in baseline_inputs
):
    print("FAIL — possible leakage-related input detected.")
else:
    print("PASS — no future, label, target, trend, or proxy input is used by the rule.")

Baseline inputs:
['avg_position', 'ctr_pct', 'gsc_clicks', 'gsc_impressions']

Potential leakage-related columns present in baseline:
[]

Leakage check:
PASS — no future, label, target, trend, or proxy input is used by the rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.